# Delayed Match-to-Sample with the CTM

Cognitive-neuroscience working-memory probe. Each clip has three phases:

```
[ sample ]   [ delay (D distractor frames) ]   [ probe ]
   cue C                                         probe P
```

Binary label = `match` iff `P == C`. The model must hold the cue identity in neuron state across the delay. We train at one delay length and **evaluate at multiple delays** to map the memory horizon of the architecture.

This notebook is fully self-contained: it defines its own dataset, its own per-frame CTM subclass, its own training loop. Only `ContinuousThoughtMachine` is imported from the repo.

In [ ]:
import os, sys, math, time
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

from models.ctm import ContinuousThoughtMachine

# ---- Config ----
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGE_SIZE = 32
SAMPLE_LEN = 2          # frames showing the cue
DELAY_LEN_TRAIN = 8     # delay length used during training
PROBE_LEN = 2           # frames showing the probe
N_FRAMES = SAMPLE_LEN + DELAY_LEN_TRAIN + PROBE_LEN

D_MODEL = 128
D_INPUT = 64
HEADS = 2
MEMORY_LENGTH = 8
MEMORY_HIDDEN = 8
SYNAPSE_DEPTH = 2
N_SYNCH = 32

TRAINING_ITERATIONS = 1500
BATCH_SIZE = 16
LR = 3e-4
SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)
print('device:', DEVICE, ' n_frames=', N_FRAMES)

## Dataset

16 distinct cues = 4 shapes × 4 colours. Distractors during the delay are uniformly random shape/colour, so the cue can occasionally re-appear by chance — but the model can't use that as a shortcut because the supervision is only about the *probe*.

In [ ]:
SHAPES = ['circle', 'square', 'triangle', 'cross']
COLORS = np.array([[1.0, 0.2, 0.2], [0.2, 1.0, 0.2], [0.3, 0.4, 1.0], [1.0, 0.9, 0.2]], dtype=np.float32)

def _draw(canvas, shape_id, color, cx, cy, radius):
    H, W = canvas.shape[1:]
    ys, xs = np.mgrid[0:H, 0:W]
    dx, dy = xs - cx, ys - cy
    if shape_id == 0:
        mask = dx*dx + dy*dy <= radius*radius
    elif shape_id == 1:
        mask = (np.abs(dx) <= radius) & (np.abs(dy) <= radius)
    elif shape_id == 2:
        mask = (dy <= radius) & (dy >= -radius) & (np.abs(dx) <= (radius - dy)/2 + 1)
    else:
        bar = max(1, radius // 3)
        mask = ((np.abs(dx) <= radius) & (np.abs(dy) <= bar)) | ((np.abs(dy) <= radius) & (np.abs(dx) <= bar))
    for c in range(3):
        canvas[c][mask] = color[c]

class DMSShapes(Dataset):
    def __init__(self, n_samples, sample_len, delay_len, probe_len, image_size, seed_offset=0):
        self.n_samples = n_samples
        self.sample_len = sample_len; self.delay_len = delay_len; self.probe_len = probe_len
        self.image_size = image_size; self.seed_offset = seed_offset
        self.n_frames = sample_len + delay_len + probe_len
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        rng = np.random.default_rng(idx + self.seed_offset)
        H = W = self.image_size
        radius = H // 6
        cue_shape = int(rng.integers(0, 4))
        cue_color = int(rng.integers(0, len(COLORS)))
        is_match = bool(rng.integers(0, 2))
        if is_match:
            probe_shape, probe_color = cue_shape, cue_color
        else:
            probe_shape, probe_color = cue_shape, cue_color
            while probe_shape == cue_shape and probe_color == cue_color:
                probe_shape = int(rng.integers(0, 4))
                probe_color = int(rng.integers(0, len(COLORS)))
        clip = np.zeros((self.n_frames, 3, H, W), dtype=np.float32)
        for t in range(self.n_frames):
            for c in range(3):
                clip[t, c] = float(rng.uniform(0, 0.15))  # subtle background tint
        cx_cue, cy_cue = rng.uniform(radius, W-radius), rng.uniform(radius, H-radius)
        for t in range(self.sample_len):
            _draw(clip[t], cue_shape, COLORS[cue_color], cx_cue, cy_cue, radius)
        for t in range(self.sample_len, self.sample_len + self.delay_len):
            ds = int(rng.integers(0, 4)); dc = int(rng.integers(0, len(COLORS)))
            cx, cy = rng.uniform(radius, W-radius), rng.uniform(radius, H-radius)
            _draw(clip[t], ds, COLORS[dc], cx, cy, radius)
        cx_p, cy_p = rng.uniform(radius, W-radius), rng.uniform(radius, H-radius)
        for t in range(self.sample_len + self.delay_len, self.n_frames):
            _draw(clip[t], probe_shape, COLORS[probe_color], cx_p, cy_p, radius)
        clip = (clip - 0.5) / 0.5
        return torch.from_numpy(clip), int(is_match)

train_data = DMSShapes(2048, SAMPLE_LEN, DELAY_LEN_TRAIN, PROBE_LEN, IMAGE_SIZE, seed_offset=0)
test_data  = DMSShapes(512,  SAMPLE_LEN, DELAY_LEN_TRAIN, PROBE_LEN, IMAGE_SIZE, seed_offset=10_000_000)
print('train', len(train_data), ' test', len(test_data))

In [ ]:
# Show a match clip and a non-match clip.
fig, axes = plt.subplots(2, N_FRAMES, figsize=(N_FRAMES*1.2, 2.6))
for row, want_match in enumerate([1, 0]):
    idx = 0
    while True:
        clip, label = train_data[idx]
        if label == want_match: break
        idx += 1
    img = clip.numpy() * 0.5 + 0.5
    for t in range(N_FRAMES):
        ax = axes[row, t]; ax.axis('off')
        ax.imshow(np.clip(np.moveaxis(img[t], 0, -1), 0, 1))
        if row == 0:
            phase = 'sample' if t < SAMPLE_LEN else ('delay' if t < SAMPLE_LEN+DELAY_LEN_TRAIN else 'probe')
            ax.set_title(f'{phase}\n t={t}', fontsize=7)
    axes[row, 0].set_ylabel(['non-match', 'match'][want_match], rotation=0, labelpad=30, fontsize=10)
fig.suptitle('Match vs non-match clip (cue → distractors → probe)', fontsize=11)
fig.tight_layout(); plt.show()

## Per-frame CTM

Subclass of `ContinuousThoughtMachine` that runs the backbone over every frame in parallel and routes attention so that internal tick `t` reads from frame `t // iterations_per_frame`.

In [ ]:
class CTMPerFrame(ContinuousThoughtMachine):
    def __init__(self, n_frames, iterations_per_frame=1, **kwargs):
        kwargs['iterations'] = n_frames * iterations_per_frame
        super().__init__(**kwargs)
        self.n_frames = n_frames
        self.iterations_per_frame = iterations_per_frame
    def compute_features(self, x):
        B, T, C, H, W = x.shape
        flat = x.reshape(B*T, C, H, W)
        flat = self.initial_rgb(flat)
        feats = self.backbone(flat)
        pos_emb = self.positional_embedding(feats)
        combined = feats + pos_emb
        _, Cp, Hp, Wp = feats.shape
        combined = combined.flatten(2).transpose(1, 2)
        kv = self.kv_proj(combined)
        self.kv_spatial_shape = (Hp, Wp)
        return kv.reshape(B, T, Hp*Wp, self.d_input)
    def forward(self, x, track=False):
        B = x.size(0); device = x.device
        kv_all = self.compute_features(x)
        state_trace = self.start_trace.unsqueeze(0).expand(B, -1, -1)
        activated_state = self.start_activated_state.unsqueeze(0).expand(B, -1)
        predictions = torch.empty(B, self.out_dims, self.iterations, device=device)
        certainties = torch.empty(B, 2, self.iterations, device=device)
        self.decay_params_action.data = torch.clamp(self.decay_params_action, 0, 15)
        self.decay_params_out.data = torch.clamp(self.decay_params_out, 0, 15)
        r_action = torch.exp(-self.decay_params_action).unsqueeze(0).repeat(B, 1)
        r_out    = torch.exp(-self.decay_params_out).unsqueeze(0).repeat(B, 1)
        d_alpha_a, d_beta_a = None, None
        _, d_alpha_o, d_beta_o = self.compute_synchronisation(activated_state, None, None, r_out, 'out')
        post_acts, attns, sync_outs, sync_acts = [], [], [], []
        for t in range(self.iterations):
            f = t // self.iterations_per_frame
            kv = kv_all[:, f]
            sync_a, d_alpha_a, d_beta_a = self.compute_synchronisation(activated_state, d_alpha_a, d_beta_a, r_action, 'action')
            q = self.q_proj(sync_a).unsqueeze(1)
            attn_out, attn_w = self.attention(q, kv, kv, average_attn_weights=False, need_weights=True)
            attn_out = attn_out.squeeze(1)
            pre = torch.cat((attn_out, activated_state), dim=-1)
            state = self.synapses(pre)
            state_trace = torch.cat((state_trace[:, :, 1:], state.unsqueeze(-1)), dim=-1)
            activated_state = self.trace_processor(state_trace)
            sync_o, d_alpha_o, d_beta_o = self.compute_synchronisation(activated_state, d_alpha_o, d_beta_o, r_out, 'out')
            predictions[..., t] = self.output_projector(sync_o)
            certainties[..., t] = self.compute_certainty(predictions[..., t])
            if track:
                post_acts.append(activated_state.detach().cpu().numpy())
                attns.append(attn_w.detach().cpu().numpy())
                sync_outs.append(sync_o.detach().cpu().numpy())
                sync_acts.append(sync_a.detach().cpu().numpy())
        if track:
            return predictions, certainties, np.array(post_acts), np.array(attns), np.array(sync_outs), np.array(sync_acts), self.kv_spatial_shape
        return predictions, certainties

def build_model(n_frames):
    return CTMPerFrame(
        n_frames=n_frames, iterations_per_frame=1,
        d_model=D_MODEL, d_input=D_INPUT, heads=HEADS,
        n_synch_out=N_SYNCH, n_synch_action=N_SYNCH,
        synapse_depth=SYNAPSE_DEPTH, memory_length=MEMORY_LENGTH,
        deep_nlms=True, memory_hidden_dims=MEMORY_HIDDEN,
        do_layernorm_nlm=False,
        backbone_type='resnet18-1', positional_embedding_type='none',
        out_dims=2, prediction_reshaper=[-1],
        dropout=0.0, neuron_select_type='random-pairing',
    ).to(DEVICE)

model = build_model(N_FRAMES)
dummy, _ = train_data[0]
model(dummy.unsqueeze(0).to(DEVICE))
print('params:', sum(p.numel() for p in model.parameters()))

## Loss & training

We use the same min-CE + max-certainty mixture from the repo, but **only over the probe-phase ticks**. Predictions during sample/delay are deliberately not supervised — the model is allowed to be uncertain there.

In [ ]:
def dms_loss(predictions, certainties, targets, probe_start):
    p = predictions[..., probe_start:]      # (B, 2, probe_len)
    c = certainties[..., probe_start:]      # (B, 2, probe_len)
    tgt = targets.unsqueeze(-1).expand(-1, p.size(-1))
    losses = nn.CrossEntropyLoss(reduction='none')(p, tgt)  # (B, probe_len)
    idx_min  = losses.argmin(dim=1)
    idx_cert = c[:, 1].argmax(dim=1)
    bi = torch.arange(p.size(0), device=p.device)
    loss = (losses[bi, idx_min] + losses[bi, idx_cert]).mean() / 2
    return loss, idx_cert + probe_start

PROBE_START = SAMPLE_LEN + DELAY_LEN_TRAIN

def train(model, train_data, training_iterations, log_every=100):
    loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    history = []
    it = iter(loader)
    t0 = time.time()
    for step in range(training_iterations):
        try:
            clips, labels = next(it)
        except StopIteration:
            it = iter(loader); clips, labels = next(it)
        clips = clips.to(DEVICE); labels = labels.to(DEVICE)
        preds, certs = model(clips)
        loss, where = dms_loss(preds, certs, labels, PROBE_START)
        opt.zero_grad(); loss.backward(); opt.step()
        if step % log_every == 0:
            with torch.no_grad():
                bi = torch.arange(preds.size(0), device=preds.device)
                acc = (preds.argmax(1)[bi, where] == labels).float().mean().item()
            history.append((step, float(loss.item()), acc))
            print(f'step {step:5d}  loss={loss.item():.3f}  acc={acc:.3f}  ({time.time()-t0:.1f}s)')
    return history

history = train(model, train_data, TRAINING_ITERATIONS)

## Memory horizon

Train at delay D, evaluate at delays {2, 4, 8, 16, 32}. The architecture's working-memory horizon is the largest D at which accuracy remains well above chance (50%).

In [ ]:
@torch.no_grad()
def evaluate_at_delay(model, delay_len, n_samples=512):
    """Build a fresh CTMPerFrame with iterations matching the new clip length, copy weights."""
    n_frames = SAMPLE_LEN + delay_len + PROBE_LEN
    eval_model = build_model(n_frames)
    eval_model.load_state_dict(model.state_dict(), strict=False)
    eval_model.eval()
    ds = DMSShapes(n_samples, SAMPLE_LEN, delay_len, PROBE_LEN, IMAGE_SIZE,
                   seed_offset=20_000_000 + delay_len)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    correct = total = 0
    probe_start = SAMPLE_LEN + delay_len
    for clips, labels in loader:
        clips = clips.to(DEVICE); labels = labels.to(DEVICE)
        preds, certs = eval_model(clips)
        idx_cert = certs[:, 1, probe_start:].argmax(dim=1) + probe_start
        bi = torch.arange(preds.size(0), device=DEVICE)
        correct += (preds.argmax(1)[bi, idx_cert] == labels).sum().item()
        total   += labels.size(0)
    return correct / total

DELAYS = [2, 4, 8, 16, 32]
horizon = {D: evaluate_at_delay(model, D) for D in DELAYS}
for D, acc in horizon.items():
    print(f'delay {D:3d}  acc {acc:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(horizon.keys()), list(horizon.values()), 'o-', color='steelblue', lw=2, markersize=8)
ax.axhline(0.5, color='grey', ls='--', alpha=0.6, label='chance (50%)')
ax.axvline(DELAY_LEN_TRAIN, color='red', ls=':', alpha=0.6, label=f'training delay = {DELAY_LEN_TRAIN}')
ax.set_xscale('log', base=2)
ax.set_xlabel('delay (frames)'); ax.set_ylabel('match/non-match accuracy')
ax.set_ylim(0.4, 1.02)
ax.set_title('Memory horizon: accuracy vs delay length')
ax.legend(); fig.tight_layout(); plt.show()

## How does the model remember?

Run a single match clip with `track=True`, then look at:
1. Per-tick certainty across the three phases.
2. Neuron raster, sorted by activity *during the delay* — surfaces the neurons that hold the cue.
3. Sync-accumulator magnitude across phases.

In [ ]:
# Pick a match clip.
for idx in range(len(test_data)):
    clip, label = test_data[idx]
    if label == 1: break
model.eval()
with torch.inference_mode():
    preds, certs, post_acts, attns, sync_outs, sync_acts, kv_shape = model(
        clip.unsqueeze(0).to(DEVICE), track=True)
preds = preds.cpu().numpy()[0]
certs = certs.cpu().numpy()[0]
post_acts = post_acts[:, 0]      # (ticks, D)
sync_outs = sync_outs[:, 0]      # (ticks, S)
sync_acts = sync_acts[:, 0]
print('clip label:', label, '  argmax at last tick:', preds[:, -1].argmax())

In [ ]:
T = certs.shape[1]
fig, ax = plt.subplots(figsize=(11, 3))
ax.axvspan(0, SAMPLE_LEN, color='gold', alpha=0.18, label='sample')
ax.axvspan(SAMPLE_LEN, SAMPLE_LEN+DELAY_LEN_TRAIN, color='lightgrey', alpha=0.35, label='delay (distractors)')
ax.axvspan(SAMPLE_LEN+DELAY_LEN_TRAIN, T, color='lightgreen', alpha=0.25, label='probe')
ax.plot(np.arange(T), certs[1], 'k-', lw=1.7)
ax.set_xlim(0, T); ax.set_ylim(-0.02, 1.02)
ax.set_xlabel('internal tick (= frame index)'); ax.set_ylabel('1 - normalised entropy')
ax.set_title('Certainty across phases — should be low until probe')
ax.legend(loc='upper left'); fig.tight_layout(); plt.show()

In [ ]:
# Neuron raster sorted by *delay-phase activity magnitude*.
delay_slice = post_acts[SAMPLE_LEN:SAMPLE_LEN+DELAY_LEN_TRAIN]    # (delay, D)
delay_score = np.abs(delay_slice).mean(axis=0)                    # (D,)
order = np.argsort(delay_score)[::-1]
raster = post_acts[:, order].T   # (D, ticks)
fig, ax = plt.subplots(figsize=(11, 6))
vmax = np.abs(raster).max()
im = ax.imshow(raster, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax, interpolation='nearest')
ax.axvline(SAMPLE_LEN-0.5, color='black', lw=0.7)
ax.axvline(SAMPLE_LEN+DELAY_LEN_TRAIN-0.5, color='black', lw=0.7)
ax.text(SAMPLE_LEN/2, -2, 'sample', ha='center', fontsize=9)
ax.text(SAMPLE_LEN+DELAY_LEN_TRAIN/2, -2, 'delay', ha='center', fontsize=9)
ax.text(SAMPLE_LEN+DELAY_LEN_TRAIN+PROBE_LEN/2, -2, 'probe', ha='center', fontsize=9)
ax.set_xlabel('tick'); ax.set_ylabel('neuron (sorted by delay activity)')
ax.set_title('Neuron raster — top rows are the "memory" neurons')
plt.colorbar(im, ax=ax, shrink=0.7); fig.tight_layout(); plt.show()

In [ ]:
sa = np.sqrt((sync_acts ** 2).mean(axis=-1))
so = np.sqrt((sync_outs ** 2).mean(axis=-1))
fig, ax = plt.subplots(figsize=(11, 3))
ax.axvspan(0, SAMPLE_LEN, color='gold', alpha=0.18)
ax.axvspan(SAMPLE_LEN, SAMPLE_LEN+DELAY_LEN_TRAIN, color='lightgrey', alpha=0.35)
ax.axvspan(SAMPLE_LEN+DELAY_LEN_TRAIN, T, color='lightgreen', alpha=0.25)
ax.plot(sa, label='action sync RMS', color='tab:orange')
ax.plot(so, label='output sync RMS', color='tab:blue')
ax.set_xlabel('tick'); ax.set_ylabel('||sync||_rms')
ax.set_title('Sync magnitudes — the leaky-integrator memory channel')
ax.legend(); fig.tight_layout(); plt.show()

## What to look at

- **Memory-horizon plot**: if accuracy stays high at delays *longer than the training delay*, the model has learned a memory mechanism rather than a fixed recipe. If it drops off sharply past the training delay, generalisation in time is limited.
- **Certainty curve**: the ideal trace stays near 0 until the probe, then jumps. A model that commits during the delay is hallucinating.
- **Top-row neurons in the raster**: these are the candidate "memory cells." Run the same clip with a *different* cue and see if the same rows come up — if yes, they are general-purpose memory; if not, they are cue-specific.

Increase `TRAINING_ITERATIONS`, `D_MODEL`, or `DELAY_LEN_TRAIN` for stronger results. Try training on a curriculum of random delays for better extrapolation.